# Pregunta 6 — Quins actors apareixen més i s'associen a més recaptació o millor nota?
**Projecte:** Anàlisi d'un conjunt de dades (0378 – Seguretat i alta disponibilitat, Tema 1)
**Dataset:** `IMDB-Movie-Data.csv` — 1.000 pel·lícules (2006–2016)

Aquest notebook treballa **una única pregunta de recerca**, amb el procediment complet: preparació de les dades, mètode d'anàlisi, resultat i interpretació.

## 1. Preparació de les dades

**Procediment:**
- Carreguem el CSV en un `DataFrame` de Pandas.
- Renombrem columnes amb espais/parèntesis (`runtime`, `revenue`).
- Convertim `Revenue = 0` a `NaN` (no és una recaptació real, és un valor desconegut).
- Separem `genre` i `actors` en llistes (cada pel·lícula en pot tenir diversos).
- Creem un subconjunt `df_fiable` amb `votes >= 1000`, perquè les pel·lícules amb pocs vots tenen una nota poc representativa (sobretot les de 2016).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("IMDB-Movie-Data.csv", encoding="utf-8")
df = df.rename(columns={"Runtime (Minutes)": "runtime", "Revenue (Millions)": "revenue"})
df.columns = [c.lower() for c in df.columns]

df.loc[df["revenue"] == 0, "revenue"] = np.nan

df["genres"] = df["genre"].str.split(",").apply(lambda l: [g.strip() for g in l])
df["actors_list"] = df["actors"].str.split(",").apply(lambda l: [a.strip() for a in l])

df_fiable = df[df["votes"] >= 1000].copy()

print(f"Total de pel·lícules: {len(df)}")
print(f"Amb >= 1000 vots (notes fiables): {len(df_fiable)}")
df.head(3)

## 2. Pregunta de recerca

**Quins actors apareixen en més pel·lícules del top 1000, i la seva presència s'associa a més recaptació o millor rating?**

**Mètode:** fem servir `df_fiable` (≥1000 vots). Desglossem la llista `actors_list` (explode) per tenir una fila per cada combinació pel·lícula–actor, i agrupem per actor calculant el nombre de pel·lícules, el rating mitjà i la recaptació mitjana. Primer mirem qui apareix més vegades; després, entre els actors amb prou pel·lícules, qui té millor recaptació mitjana.

In [ ]:
a = df_fiable.explode("actors_list")
resum_actor = (a.groupby("actors_list")
                 .agg(n=("title", "count"),
                      rating_mitja=("rating", "mean"),
                      revenue_mitja=("revenue", "mean"))
                 .round(2))
print(f"Actors diferents al dataset: {len(resum_actor)}")

### Actors que apareixen en més pel·lícules

In [ ]:
top_n = resum_actor.sort_values("n", ascending=False).head(10)
top_n

### Entre els que surten més (≥8 films), els de millor recaptació mitjana

In [ ]:
resum_actor.query("n >= 8").sort_values("revenue_mitja", ascending=False).head(5)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
top_n["n"][::-1].plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Actors amb més pel·lícules al top 1000")
ax.set_xlabel("Nombre de pel·lícules")
plt.tight_layout(); plt.show()

## 3. Resultats

- Els actors amb més pel·lícules al dataset són **Mark Wahlberg** (15), **Hugh Jackman** (14), **Brad Pitt** i **Christian Bale** (13 cadascun).
- Aparèixer sovint no implica millor recaptació mitjana: Mark Wahlberg (15 films) té una recaptació mitjana de només 99,9 M, mentre que **Robert Downey Jr.** (12 films) arriba a 262,3 M de mitjana i **Jennifer Lawrence** (11 films) a 219,2 M, gràcies al seu paper en grans sagas (Marvel, Hunger Games).
- En rating, Robert Downey Jr. també destaca amb un 7,50 de mitjana, el més alt entre els actors amb més aparicions.


## 4. Interpretació i limitacions

✏️ **Omple-ho tu:** Aparèixer sovint no es tradueix en més recaptació perquè depèn del tipus de projecte: Downey Jr. o Lawrence concentren la seva filmografia en grans sagues (Marvel, Hunger Games), que per si soles garanteixen taquilla. És la franquícia qui empeny els números, més que l’actor en si.